# Motion-Guided Self-Supervised Learning for Cardiac Cine MRI
## Notebook 07: Limited-Label Fine-Tuning & Label-Efficiency Experiments

This notebook provides a comprehensive walkthrough of the **Limited-Label Fine-Tuning and Evaluation Pipeline**.

### Experimental Matrix (16 Configurations):
- **4 Annotation Regimes (Patient-Level)**:
  - **10% Labels**: 7 patients (128 slices)
  - **25% Labels**: 17 patients (318 slices)
  - **50% Labels**: 35 patients (660 slices)
  - **100% Labels**: 70 patients (1,324 slices)
- **4 Modular Model Variants**:
  - **Variant A (supervised)**: 2D U-Net baseline initialized from scratch
  - **Variant B (ssl_finetune)**: 2D U-Net initialized with SSL pretrained SharedEncoder
  - **Variant C (ssl_motion)**: Pretrained U-Net regularized with motion consistency
  - **Variant D (ssl_motion_pseudo)**: Progressive framework with confidence-filtered pseudo-labels

> **COMPUTE CONSTRAINT NOTE**:
> Per project requirements, actual multi-epoch experiments are run on a dedicated GPU training server (`TRAINING MACHINE ONLY`).
> This notebook validates subset integrity, model instantiation, multi-component losses, and experiment tracking on CPU.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from src.segmentation_model import SegmentationUNet
from src.encoder import load_encoder_weights, count_parameters
from src.motion import MotionEstimator
from src.dataset import ACDCSegDataset, ACDCTemporalDataset
from src.experiment_runner import build_experiment_model, ExperimentLossManager, ExperimentRegistry

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")

### 1. Patient-Level Split Verification & Strict Nesting
The project enforces patient-level sampling with seed 42 to prevent slice leakage across training, validation, and testing.
Subsets are strictly nested: $10\% \subset 25\% \subset 50\% \subset 100\%$.

In [ ]:
splits_dir = project_root / "data" / "splits"
val_pts = set(open(splits_dir / "val_patients.txt").read().split())
test_pts = set(open(splits_dir / "test_patients.txt").read().split())

fractions = [10, 25, 50, 100]
patient_subsets = {}
slice_counts = {}

for pct in fractions:
    split_file = splits_dir / f"labeled_{pct}.txt"
    pts = [line.strip() for line in open(split_file) if line.strip() and not line.startswith('#')]
    patient_subsets[pct] = set(pts)
    
    # Count slices
    dataset = ACDCSegDataset(
        processed_dir=str(project_root / "data" / "processed"),
        split_file=str(split_file),
    )
    slice_counts[pct] = len(dataset)
    print(f"Subset labeled_{pct:3d}.txt: {len(pts):2d} patients | {len(dataset):4d} labeled slices")

# Verify nesting
assert patient_subsets[10].issubset(patient_subsets[25]), "10% not subset of 25%!"
assert patient_subsets[25].issubset(patient_subsets[50]), "25% not subset of 50%!"
assert patient_subsets[50].issubset(patient_subsets[100]), "50% not subset of 100%!"

# Verify zero leakage
for pct in fractions:
    assert len(patient_subsets[pct].intersection(val_pts)) == 0, f"Leakage into validation!"
    assert len(patient_subsets[pct].intersection(test_pts)) == 0, f"Leakage into test!"

print("\nSplit integrity checks verified: Strict patient nesting and zero cross-split leakage.")

### 2. Visualizing Patient and Slice Counts Across Regimes
We plot the patient and slice distribution across the 4 label regimes.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

pct_labels = [f"{p}%" for p in fractions]
pt_counts = [len(patient_subsets[p]) for p in fractions]
sl_counts = [slice_counts[p] for p in fractions]

ax1.bar(pct_labels, pt_counts, color='royalblue', alpha=0.85, edgecolor='black')
ax1.set_title("Labeled Patients per Regime")
ax1.set_ylabel("Number of Patients")
ax1.grid(axis='y', linestyle='--', alpha=0.7)
for i, v in enumerate(pt_counts):
    ax1.text(i, v + 1, f"{v} pts", ha='center', fontweight='bold')

ax2.bar(pct_labels, sl_counts, color='seagreen', alpha=0.85, edgecolor='black')
ax2.set_title("Labeled Slices per Regime")
ax2.set_ylabel("Number of 2D Slices")
ax2.grid(axis='y', linestyle='--', alpha=0.7)
for i, v in enumerate(sl_counts):
    ax2.text(i, v + 25, f"{v} slices", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 3. Inspecting the Parameterized Experiment Configuration
We load `configs/experiments.yaml` defining parameters for the 4 variants.

In [ ]:
config_path = project_root / "configs" / "experiments.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Configured Model Variants:")
for var_name, var_info in config['variants'].items():
    print(f"  - {var_name:18s}: {var_info['description']}")

print(f"\nLoss Weights:")
print(f"  Dice weight:   {config['loss']['dice_weight']}")
print(f"  CE weight:     {config['loss']['ce_weight']}")
print(f"  Motion weight: {config['loss']['motion_weight']}")
print(f"  Pseudo weight: {config['loss']['pseudo_weight']}")

### 4. Demonstrating Model Initialization & Pretrained Encoder Transfer
We instantiate `SegmentationUNet` and demonstrate loading the SSL-pretrained `SharedEncoder` weights.

In [ ]:
device = torch.device('cpu')
ssl_ckpt = project_root / "checkpoints" / "ssl" / "ssl_encoder_best.pth"

model, motion_est = build_experiment_model(
    config=config,
    mode="ssl_motion_pseudo",
    ssl_checkpoint=str(ssl_ckpt) if ssl_ckpt.exists() else None,
    device=device,
)

print(f"SegmentationUNet Parameters: {count_parameters(model):,}")
print(f"MotionEstimator Parameters:   {count_parameters(motion_est):,}")

### 5. Multi-Component Loss Demonstration
We demonstrate the combined loss computation:
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{sup}} + \lambda_{\text{motion}} \mathcal{L}_{\text{motion}} + \lambda_{\text{pseudo}} \mathcal{L}_{\text{pseudo}}$$

In [ ]:
loss_manager = ExperimentLossManager(
    dice_weight=1.0,
    ce_weight=1.0,
    motion_weight=0.1,
    pseudo_weight=0.25,
    num_classes=4,
)

# Load sample batch
seg_dataset = ACDCSegDataset(
    processed_dir=str(project_root / "data" / "processed"),
    split_file=str(splits_dir / "labeled_10.txt"),
)
sample = seg_dataset[0]
img = sample['image'].unsqueeze(0)
target = sample['mask'].unsqueeze(0)

model.eval()
with torch.no_grad():
    logits = model(img)

# Compute supervised loss
total_loss, breakdown = loss_manager.compute_loss(
    logits_sup=logits,
    targets_sup=target,
)

print("Loss Breakdown on Sample:")
for k, v in breakdown.items():
    print(f"  {k:15s}: {v:.4f}")

### 6. Inspecting the Experiment Registry
All experiment runs are automatically tracked in `results/experiments/experiment_registry.json`.

In [ ]:
registry = ExperimentRegistry(project_root / "results" / "experiments" / "experiment_registry.json")
print(f"Total Registered Experiments: {len(registry.records)}")

for exp_id, rec in list(registry.records.items())[:4]:
    print(f"  [{exp_id}]: Mode={rec.get('mode')} | Fraction={rec.get('label_fraction')}% | Status={rec.get('status')}")

### 7. Training Matrix Commands (`TRAINING MACHINE ONLY`)

To execute the full experimental matrix on the separate GPU training system:

```bash
# ========================================================
# TRAINING MACHINE ONLY (DO NOT RUN ON DEV SYSTEM)
# ========================================================

# Variant A: Supervised Baseline across 10%, 25%, 50%, 100%
python src/experiment_runner.py --mode supervised --label-fraction 10 --device cuda
python src/experiment_runner.py --mode supervised --label-fraction 25 --device cuda
python src/experiment_runner.py --mode supervised --label-fraction 50 --device cuda
python src/experiment_runner.py --mode supervised --label-fraction 100 --device cuda

# Variant B: SSL Fine-Tuning across label regimes
python src/experiment_runner.py --mode ssl_finetune --label-fraction 10 --device cuda
python src/experiment_runner.py --mode ssl_finetune --label-fraction 25 --device cuda
python src/experiment_runner.py --mode ssl_finetune --label-fraction 50 --device cuda
python src/experiment_runner.py --mode ssl_finetune --label-fraction 100 --device cuda

# Variant C: SSL + Motion Consistency across label regimes
python src/experiment_runner.py --mode ssl_motion --label-fraction 10 --device cuda
python src/experiment_runner.py --mode ssl_motion --label-fraction 25 --device cuda
python src/experiment_runner.py --mode ssl_motion --label-fraction 50 --device cuda
python src/experiment_runner.py --mode ssl_motion --label-fraction 100 --device cuda

# Variant D: Progressive Framework (SSL + Motion + Pseudo-Labels)
python src/experiment_runner.py --mode ssl_motion_pseudo --label-fraction 10 --device cuda
python src/experiment_runner.py --mode ssl_motion_pseudo --label-fraction 25 --device cuda
python src/experiment_runner.py --mode ssl_motion_pseudo --label-fraction 50 --device cuda
python src/experiment_runner.py --mode ssl_motion_pseudo --label-fraction 100 --device cuda
```